# RAFT · Stage 2 — Fine-tune the model

Supervised fine-tuning (LoRA under the hood) of the fine-tuned model `gpt-4.1-mini` on the RAFT
dataset from Stage 1. Uploads train/val, submits the job, polls to completion, prints the
fine-tuned model id.

**Runtime:** ~1.5 h. **Cost:** training is a few USD (tokens × epochs × price); the deployment
hosting fee (Stage 3) is what actually matters — use the Developer tier there.

Two ways to run this stage — the **Foundry portal wizard** (Option A) is the fastest, the
**SDK cells below** (Option B) are the reproducible / papermill path. Both produce the same
fine-tuned model id for Stage 3.

Fine-tuning access requires the **Cognitive Services OpenAI Contributor** role on the Foundry
(Azure OpenAI) resource; deploying the result also needs `deployments/write` (Foundry Owner).
Verify regional quota before running. Papermill-compatible; no state from Stage 1's kernel.

Docs: [Customize a model with fine-tuning](https://learn.microsoft.com/azure/foundry/openai/how-to/fine-tuning).

## Option A — Foundry portal (fastest)

The quickest path is the **Foundry (new)** portal wizard — no SDK. Full walkthrough with
screenshots: [`docs/raft-finetune-foundry-ui.md`](../../docs/raft-finetune-foundry-ui.md).
Steps (verified Aug 2026):

1. Sign in to [Microsoft Foundry](https://ai.azure.com/) with the **New Foundry** toggle **on**.
2. Select your subscription and Foundry resource.
3. Go to **Build → Fine-tune** and select **Start fine-tuning**.
4. Select the base model (`gpt-4.1-mini`) — or a previously fine-tuned model for continuous
   fine-tuning (`base-model.ft-{jobid}`).
5. Choose the customization method: **Supervised (SFT)** for RAFT.
6. Choose the training type: **Developer** while iterating (preemptible, ~50% off), **Global**
   for the final run, **Standard** if you need in-region data residency.
7. Upload / select the datasets `data/raft_train.jsonl` and `data/raft_val.jsonl` (JSONL, chat
   format, UTF-8 with BOM — produced by Stage 1).
8. Optionally set a **suffix** (`raft-aml`), **seed** and hyperparameters (e.g. `n_epochs = 2`).
9. Select **Submit**.

After the job completes: review training metrics and checkpoints, **confirm the safety
evaluation status**, then **Deploy** from the job details page (that is Stage 3). Enabling
**auto-deploy** on success removes a manual step (OpenAI models only). Copy the fine-tuned model
id into `data/ft_model_id.txt` (or the `ft_model_id` parameter in Stage 3) and set the Terraform
var `raft_student_ft_model_id`.

## Option B — SDK (reproducible / papermill)

The cells below do the same via the SDK so the run is scriptable and non-interactive.

In [ ]:
region = "swedencentral"
student_model = "gpt-4.1-mini"
training_tier = "Developer"  # trainingType: Developer (iterate, ~50% off, preemptible) | GlobalStandard (final) | Standard (data residency)
n_epochs = 2
train_path = "data/raft_train.jsonl"
val_path = "data/raft_val.jsonl"
ft_model_out = "data/ft_model_id.txt"  # written for Stage 3 / Terraform var raft_student_ft_model_id

In [ ]:
import os, time, pathlib
FOUNDRY_ENDPOINT = os.environ.get("AI_FOUNDRY_ENDPOINT", "")
t0 = time.time()
print(f"Stage 2 · model={student_model} · tier={training_tier} · epochs={n_epochs} · region={region}")
assert FOUNDRY_ENDPOINT, "AI_FOUNDRY_ENDPOINT must be set (see infra/terraform outputs)."

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")
# 2025-04-01-preview is required for the trainingType (Developer/Global/Standard) tier selector.
client = AzureOpenAI(azure_endpoint=FOUNDRY_ENDPOINT, azure_ad_token_provider=token_provider, api_version="2025-04-01-preview")

def upload(path):
    with open(path, "rb") as f:
        return client.files.create(file=f, purpose="fine-tune").id

train_file = upload(train_path)
val_file = upload(val_path)
print(f"uploaded train={train_file} val={val_file}")

In [ ]:
# Submit the SFT job (current OpenAI SDK surface): hyperparameters go under `method`, and the
# training tier under extra_body `trainingType`. Continuous fine-tuning: pass a prior ft model as
# `model` (base-model.ft-{jobid}) to chain jobs — this is how the retraining loop becomes real.
job = client.fine_tuning.jobs.create(
    model=student_model,
    training_file=train_file,
    validation_file=val_file,
    suffix="raft-aml",
    method={"type": "supervised", "supervised": {"hyperparameters": {"n_epochs": n_epochs}}},
    extra_body={"trainingType": training_tier},  # Developer | GlobalStandard | Standard
)
print(f"job {job.id} status={job.status}")

In [ ]:
# Poll to completion (~1.5 h). Every epoch produces a deployable checkpoint — keep the last three.
terminal = {"succeeded", "failed", "cancelled"}
while True:
    job = client.fine_tuning.jobs.retrieve(job.id)
    print(f"{time.strftime('%H:%M:%S')} status={job.status}")
    if job.status in terminal:
        break
    time.sleep(60)

assert job.status == "succeeded", f"fine-tune {job.status}"
ft_id = job.fine_tuned_model
pathlib.Path(ft_model_out).write_text(ft_id, encoding="utf-8")
print(f"Stage 2 done in {(time.time()-t0)/60:.1f} min. ft model: {ft_id}")
print(f"Set Terraform var raft_student_ft_model_id={ft_id}, then run 3_deploy.ipynb")